In [ ]:
import os
import random
from typing import Dict

import matplotlib.pyplot as plt
import numpy as np
import scipy
import torch
from sklearn.metrics import r2_score
from tqdm import tqdm

import barostat_parameters
from graph_utils import prepare_traj
from itpo_weights import DatasetType
from pressure import compute_total_stress
from simulator_model import Model as VelocityModel
from training_utils import freeze_normalizer
from utils import (
    build_velocity_graph_correction,
    calc_p_ratio_box_tensor,
    load_and_split_dataset,
    rollout_cascade,
)


### Data

In [ ]:
poisson_buckets = [
    {"max": 0.1, "count": 100},                # P < 0.1
    {"min": 0.1, "max": 0.2, "count": 100},    # 0.1 <= P < 0.2
    {"min": 0.2, "count": 200}                 # P >= 0.2
]

dataset_type = DatasetType.NodeOptimized

train_files, val_files, test_files = load_and_split_dataset(
    registry_path="./data_mini/data_registry_mini.csv",
    target_data_type=dataset_type,
    possion_buckets=poisson_buckets,
    split_ratios=(0.5, 0.25, 0.25),
    seed=42
)

# Load actual data
data = {
    'train': {},
    'val' : {},
    'test' : {},
}

print("Loading data...")
for key in data.keys():
    if key == 'train':
        data[key] = [torch.load(file, weights_only=False) for file in tqdm(train_files, desc=f"{key:<5} data")]
    elif key == 'val':
        data[key] = [torch.load(file, weights_only=False) for file in tqdm(val_files, desc=f"{key:<5} data")]
    elif key == 'test':
        data[key] = [torch.load(file, weights_only=False) for file in tqdm(test_files, desc=f"{key:<5} data")]
    else:
        raise ValueError(f"Unexpected key in data dictionary: {key}. ")

print("\nPreparing data...")
for data_type, sims in data.items():
    prepared = []
    for sim in tqdm(sims, desc=f"{data_type:<5} data"):
        prepared_sim = prepare_traj(sim, calc_angles=False)
        prepared.append(prepared_sim)
    data[data_type] = prepared

print(f"\nTrain data: {len(data['train'])} sims.")
print(f"Val data:   {len(data['val'])} sims.")
print(f"Test data:  {len(data['test'])} sims.")


#### Show $\nu$ distribution

In [ ]:
def visualize_nu_disribution(data: Dict):
    ps = {}
    all_values = []

    for data_type, sims in data.items():
        ps[data_type] = [calc_p_ratio_box_tensor(sim).item() for sim in sims]
        all_values.extend(ps[data_type])

    min_val = min(all_values)
    max_val = max(all_values)
    common_bins = np.linspace(min_val, max_val, 30) 

    for data_type, values in ps.items():
        plt.hist(
        values, 
        bins=common_bins, 
        edgecolor='black', 
        alpha=0.6, 
        label=f"{data_type} data"
    )

    plt.title("$\\nu$ distribution")
    plt.xlabel("GT LAMMPS $\\nu$")
    plt.ylabel("N")
    plt.legend()
    plt.show()

visualize_nu_disribution(data)


### Initialize pretrained cascade

In [ ]:
hidden_size = 128
mp_layers = 2
mlp = 3
epochs = 100

device = 'cuda'
model_save_path = os.path.join("./trained_models", f"{dataset_type}", "cascade", "refined")
n_models = len(os.listdir(model_save_path))

models = []
for h in tqdm(range(n_models)):
    if h == 0:
        n_graphs = 1
    else:
        n_graphs = h+1
    
    init_graph = build_velocity_graph_correction([data['val'][0][i].cpu().detach() for i in range(n_graphs)], panic_at_positions=False).to(device)

    current_h_model = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
    current_h_model.load_checkpoint(os.path.join(model_save_path, f"model_refined_h{h}_nl{mp_layers}_mlp{mlp}_epochs{epochs}.pt"))
    current_h_model = freeze_normalizer(current_h_model)
    for param in current_h_model.parameters():
        param.requires_grad = False
    models.append(current_h_model)

print(f"Loaded cascade of {len(models)} pretrained models.")


### Testing models

In [ ]:
factors = [(sim[4].box_tensor[0]/sim[3].box_tensor[0]).item() for sim in data["test"]]
mean_factor = sum(factors)/len(factors)

num_steps = 50
target_idx = num_steps

results = {
    "mse": [],
    "pred_p": [],
    "gt_p": [],
    "gt_box": [],
    "pred_box": [],
    "gt_pressure": [],
    "pred_pressure": [],
}
with torch.no_grad():
    for val_sim in tqdm(data['test']):

        sim_strain = (val_sim[1].box.x - val_sim[-1].box.x) / val_sim[0].box.x
        assumed_rollout_length = int(sim_strain / 1e-5 / 0.01)
        dump_period = int(assumed_rollout_length / len(val_sim)) + 1
        
        barostat_config = barostat_parameters.node_optimizated if dataset_type == "node_optimized" else barostat_parameters.stiff_optimized

        # Rollout
        rollout = rollout_cascade(
            models=models,
            initial_state=val_sim[0],
            num_steps=num_steps,
            barostat_config=barostat_config,
            box_compression_factor=mean_factor,
            device=device
        )

        # Compute Position MSE
        pos_mse = [torch.nn.functional.mse_loss(rollout[i].pos.cpu(), val_sim[i].to(device).pos.cpu()).item() for i in range(len(rollout))]
        results["mse"].append(pos_mse)
        
        # Compute GT and predicted Poisson's ratio
        pred_p = calc_p_ratio_box_tensor(rollout).item()
        results["pred_p"].append(pred_p)
        gt_p = calc_p_ratio_box_tensor(val_sim[:target_idx]).item()
        results["gt_p"].append(gt_p)
        
        # Save box dimensions
        pred_box = [g.box_tensor.cpu() for g in rollout]
        results["pred_box"].append(pred_box)
        gt_box = [g.box_tensor.cpu() for g in val_sim[: len(rollout)]]
        results["gt_box"].append(gt_box)
        
        # Compute GT and predicted pressure
        r0 = rollout[0].edge_attr[:, -2]
        gt_pressure = torch.stack([compute_total_stress(g, r0=r0).cpu() for g in val_sim[: len(rollout)]], dim=0)
        results['gt_pressure'].append(gt_pressure)
        pred_pressure = torch.stack([compute_total_stress(g, r0=r0).cpu() for g in rollout], dim=0)
        results["pred_pressure"].append(pred_pressure)


In [ ]:
params = {
    'font.size': 8,             # Base font size
    'axes.labelsize': 8,        # Axis labels (e.g., nu_gt)
    'axes.titlesize': 8,        # Subplot titles
    'xtick.labelsize': 7,       # Tick numbers
    'ytick.labelsize': 7,
    'legend.fontsize': 6,       # Make legend smaller to fit
    'lines.markersize': 4,      # Reduce scatter dot size
    'figure.figsize': (3.33, 3.33), # Your target size
    'figure.dpi': 200,          # High DPI for clear viewing
    'font.family': 'serif',     # Matches most LaTeX/Paper fonts
}
plt.rcParams.update(params)

fig, ax = plt.subplots(1, 1, layout="constrained")

parity_line = (min(results["gt_p"]) - 0.05, max(results["gt_p"]) + 0.05)
ax.plot(parity_line, parity_line, color="black", linewidth=1, linestyle='--')

# Parity plot
r2 = r2_score(results['gt_p'], results['pred_p'])
res = scipy.stats.spearmanr(results["gt_p"], results["pred_p"])
sp = res.statistic
label_text = f"$R^2={r2:.3f}$, SP={sp:.3f}"

ax.scatter(results["gt_p"], results["pred_p"], label=label_text)
ax.legend(frameon=False, loc='best')
ax.set_xlabel(r"$\nu_{gt}$")
ax.set_ylabel(r"$\nu_{pred}$")

plt.show()


#### Position MSE as function of GT Poisson's ratio $\nu$

In [ ]:
plt.scatter(results["gt_p"], [results['mse'][i][-1] for i in range(len(results["gt_p"]))])
plt.xlabel(r"$\nu_{GT}$")
plt.ylabel(r"Position MSE")
plt.yscale('log')
plt.show()

#### Position MSE as function of rollout step

In [ ]:
rand_sim = random.randint(0, len(results['mse'])-1)
plt.plot(results["mse"][rand_sim])

plt.legend()
plt.title(f"Sim {rand_sim}")
plt.xlabel('Rollout step')
plt.ylabel('Position MSE')
plt.yscale('log')
plt.show()

#### Box behaviour

In [ ]:
rand = random.randint(0, len(results) - 1)

box_y_true = [results["gt_box"][rand][i][1].item() for i in range(1, len(rollout) - 1)]
box_x_true = [results["gt_box"][rand][i][0].item() for i in range(1, len(rollout) - 1)]
box_y_roll = [results["pred_box"][rand][i][1].item() for i in range(1, len(rollout) - 1)]
box_x_roll = [results["pred_box"][rand][i][0].item() for i in range(1, len(rollout) - 1)]

fig, ax = plt.subplots(1, 2, sharex=True, layout='constrained', figsize=(6, 3))

ax[0].plot(box_y_roll, label="Pred")
ax[0].plot(box_y_true, label="GT")
ax[0].set_xlabel("Rollout step")
ax[0].set_ylabel("$L_y$")
ax[0].legend()

ax[1].plot(box_x_roll, label='Pred')
ax[1].plot(box_x_true, label='GT')
ax[1].set_xlabel("Rollout step")
ax[1].set_ylabel("$L_x$")
ax[1].legend()

plt.show()